In [ ]:
import librosa
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import numpy as np
import os
import random
import shutil
from keras.preprocessing.image import ImageDataGenerator
from keras.layers import (Input, Add, Dense, Activation, ZeroPadding2D, BatchNormalization, Flatten,
                          Conv2D, AveragePooling2D, MaxPooling2D, GlobalMaxPooling2D, Dropout)
from keras.models import Model, load_model
import keras.backend as K
from tensorflow.keras.optimizers import Adam

In [ ]:
# Connect Google Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Define constants

# Output folder path
output_folder_path = '/content/drive/MyDrive/prev-experiments/cnn_2/out/'

# Genres folder path
genres_folder_path = '/content/drive/MyDrive/prev-experiments/prev_experiments_datasets'

# Genres to analyze
genres = ['deep_house', 'tech_house', 'melodic_techno', 'progressive', 'techno_peak_time', 'hard_techno', 'minimal', 'trance']

# Test %
test_percentage = 0.2

In [ ]:
for i, genre in enumerate(genres): # Iterate over genres
  for j, file_name in enumerate(os.listdir(genres_folder_path + '/' + genre + '_data')): # Iterate over each file of a specific genre
    print('analyzing: genre ' + str(i) + '- track ' + str(j))
    song_name = genres_folder_path + '/' + genre + '_data/' + file_name

    # Load track using librosa
    y, sr = librosa.load(song_name, mono=True, duration=30)

    # Calculate features for the track
    mels = librosa.feature.melspectrogram(y=y,sr=sr)
    fig = plt.Figure()
    canvas = FigureCanvas(fig)
    p = plt.imshow(librosa.power_to_db(mels,ref=np.max))
    plt.savefig(f'{output_folder_path}{genre}/{j}.png')

In [ ]:
for genre in genres:
  filenames = os.listdir(output_folder_path + genre)
  random.shuffle(filenames)
  split_idx = int(len(filenames)*test_percentage)
  test_files = filenames[:split_idx]
  train_files = filenames[split_idx:]
  for fte in test_files:
    shutil.move(output_folder_path + genre + '/' + fte, output_folder_path + '/test/' + genre + '/' + fte)
  for ftr in train_files:
    shutil.move(output_folder_path + genre + '/' + ftr, output_folder_path + '/train/' + genre + '/' + ftr)

In [ ]:
train_dir = output_folder_path + 'train/'
train_datagen = ImageDataGenerator(rescale=1./255)
train_generator = train_datagen.flow_from_directory(train_dir,target_size=(288,432),color_mode="rgba",class_mode='categorical',batch_size=128)

validation_dir = train_dir = output_folder_path + 'test/'
vali_datagen = ImageDataGenerator(rescale=1./255)
vali_generator = vali_datagen.flow_from_directory(validation_dir,target_size=(288,432),color_mode='rgba',class_mode='categorical',batch_size=128)

Found 2560 images belonging to 8 classes.
Found 640 images belonging to 8 classes.


In [ ]:
def GenreModel(input_shape = (288,432,4),classes=8):

  X_input = Input(input_shape)

  X = Conv2D(8,kernel_size=(3,3),strides=(1,1))(X_input)
  X = BatchNormalization(axis=3)(X)
  X = Activation('relu')(X)
  X = MaxPooling2D((2,2))(X)

  X = Conv2D(16,kernel_size=(3,3),strides = (1,1))(X)
  X = BatchNormalization(axis=3)(X)
  X = Activation('relu')(X)
  X = MaxPooling2D((2,2))(X)

  X = Conv2D(32,kernel_size=(3,3),strides = (1,1))(X)
  X = BatchNormalization(axis=3)(X)
  X = Activation('relu')(X)
  X = MaxPooling2D((2,2))(X)

  X = Conv2D(64,kernel_size=(3,3),strides=(1,1))(X)
  X = BatchNormalization(axis=-1)(X)
  X = Activation('relu')(X)
  X = MaxPooling2D((2,2))(X)

  X = Conv2D(128,kernel_size=(3,3),strides=(1,1))(X)
  X = BatchNormalization(axis=-1)(X)
  X = Activation('relu')(X)
  X = MaxPooling2D((2,2))(X)


  X = Flatten()(X)

  X = Dropout(rate=0.3)(X)

  X = Dense(classes, activation='softmax', name='fc' + str(classes))(X)

  model = Model(inputs=X_input,outputs=X,name='GenreModel')

  return model

In [ ]:
def get_f1(y_true, y_pred): #taken from old keras source code
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
    precision = true_positives / (predicted_positives + K.epsilon())
    recall = true_positives / (possible_positives + K.epsilon())
    f1_val = 2*(precision*recall)/(precision+recall+K.epsilon())
    return f1_val

model = GenreModel(input_shape=(288,432,4),classes=len(genres))
opt = Adam(learning_rate=0.0005)
model.compile(optimizer = opt,loss='categorical_crossentropy',metrics=['accuracy',get_f1])

model.fit_generator(train_generator,epochs=70,validation_data=vali_generator)